In [1]:
import json
import os

import pandas as pd

# Dataset formation
From JSONs to unable data frame

Test format in notebook. To be transformed into .py file.

Start: 02 Sept. 2025

Last modification: 02 Sept. 2025

In [2]:
def open_file(eID: str, sID: str, data_path = 'Data/private_data/sessionInfo'):
    """ 
    Opens the JSON file of one session of an exercise.

    INPUT PARAMETERS
    * eID: Unique Exercise ID in string format.
    * sID: Unique Session ID in string format.
    * data_path: Directory path where the data is stored.

    OUTPUT PARAMETERS
    * data: the opened JSON file in str-dict format.
    """

    with open(f'{data_path}/{eID}/{sID}.json', 'r') as file:
        data = json.load(file)

    return data

In [3]:
def open_exercise_sessions(eID: str, data_path = 'Data/private_data/sessionInfo'):
    """
    Opens the JSON files of all sessions for one exercise. 
    
    INPUT PARAMETERS
    * eID: Unique Exercise ID in string format. 
    * data_path: Directory path where the data is stored.
    
    OUTPUT PARAMETERS
    * exercise_dict: Dictionary with the data of a session stored per session key.
    """
    exercise_dict = {}

    sessions = os.listdir(f'{data_path}/{eID}')
    sessions = [sID.replace(".json", "") for sID in sessions] # removing .json

    for sID in sessions:
        data_sID = open_file(eID, sID, data_path)
        exercise_dict[sID] = data_sID

    return exercise_dict

In [4]:
def open_library(data_path = 'Data/private_data/sessionInfo'):
    """
    Opens the JSON files of all sessions of all exercises (library) and stores them into one long dictionary.
    
    INPUT PARAMETERS
    * data_path: Directory path where the data is stored.
    
    OUTPUT PARAMETERS
    * library_dict: Dictionary with the data of all sessions stored per session per exercise.
    """
    library_dict = {}

    exercises = os.listdir(f'{data_path}')

    for eID in exercises:
        exercise_dict = open_exercise_sessions(eID, data_path)
        library_dict[eID] = exercise_dict

    return library_dict


In [5]:
library_dict = open_library()

In [6]:
def event_dataframes(event_dict: str, interactionID: str):
    """
    A dataframe with all events of one interaction.
    """

    event_columns_mapping = {'timestamp': 'eventID', 
                         'event': 'eventType', 
                         'tags': 'eventTags', 
                         'annotations': 'eventAnnotations', 
                         'exerciseStatus': 'exerciseStatusAfterEvent', 
                         'progress': 'progressAfterEvent',
                         'inputStatus': 'inputStatus'}
    

    df_event = pd.json_normalize(event_dict).rename(columns=event_columns_mapping)
    iIDs = len(df_event) * [interactionID]
    df_event['interactionID'] = iIDs
    df_event = df_event.iloc[:, [-1] + list(range(len(df_event.columns)-1))]
    
    return df_event


In [135]:
def interaction_dataframes(interaction_dict: str, questionID: str):
    """
    A dataframe with all interactions of one question and all events of that interaction included.
    """
    if type(interaction_dict) == dict:
        interaction_dict = [interaction_dict]

    interaction_columns_mapping = {'itemType': 'itemType',
                                   'content': 'itemContent',
                                   'id': 'interactionID',
                                   'interactionType': 'interactionType',
                                   'result': 'interactionResult',
                                   'solution': 'interactionSolution',
                                   'derivation': 'interactionDerivation'}

    # Create basis of the interaction dataframe
    df_interaction = pd.json_normalize(interaction_dict).rename(columns=interaction_columns_mapping) #flatten all contents of interactions
    df_interaction['questionID'] = len(df_interaction) * [questionID]
    df_interaction = df_interaction.iloc[:, [-1] + list(range(len(df_interaction.columns)-1))]

    # Append the event dataframe to the interactions
    for item in interaction_dict:
        
        if item['itemType'] == 'INTERACTION':
            events = item['result']['events']
            itemID = item['id']
            
            list_event_frames = []
            for event in events:
                one_item = event_dataframes(event, itemID)
                list_event_frames.append(one_item)
            
            if len(list_event_frames) >= 1 :
                df_event = pd.concat(list_event_frames, axis=0, ignore_index=True)
                df_interaction = pd.merge(df_interaction, df_event, on="interactionID", how='outer')
            
            else:
                df_interaction = df_interaction





    return df_interaction


In [136]:
def question_dataframes(questions, sessionID):
    """
    A dataframe with all questions of one session and all interactions of that question and all events of that interaction included.
    """

    question_columns_mapping = {'id': 'questionID',
                                   'type': 'questionType',
                                   'items': 'items'}

    # Create basis of the question dataframe
    df_question = pd.json_normalize(questions).rename(columns=question_columns_mapping).drop('items', axis=1) #flatten all contents of interactions
    df_question['sessionID'] = len(df_question) * [sessionID]
    df_question = df_question.iloc[:, [-1] + list(range(len(df_question.columns)-1))]
    
    list_question_frames = []
    # Append the interaction dataframe to the question
    for question in questions:
        interactions = question['items']
        questionID = question['id']

        one_question = interaction_dataframes(interactions, questionID)
        
        list_question_frames.append(one_question)

            
    df_interaction = pd.concat(list_question_frames, axis=0, ignore_index=True)

    
    # Join the interaction dataframe with the question dataframe
    df_question = pd.merge(df_question, df_interaction, on="questionID", how='outer')
        

    return df_question

In [137]:
def handle_skills(df_session):

    skill_frames = []
    for j in range(len(df_session)):
        session = df_session.iloc[j]
        df_skills = pd.DataFrame.from_dict(session['skills']).T.reset_index().rename(columns={'index': 'skillName'})
        df_skills['sessionID'] = len(df_skills) * [session['sessionID']]
        skill_frames.append(df_skills)
    
    df_skills = pd.concat(skill_frames).reset_index().drop(['index', 'descr'], axis=1)
    df_skills = df_skills.groupby(['sessionID']).agg(list).reset_index().rename(columns={'index': 'sessionID'})
    
    return df_skills

In [153]:
def session_dataframes(library_dict, exerciseID):
    """
    A dataframe with all sessions of one exercise and all questions of that session and all interactions of that question and all events of that interaction included.
    """

    question_columns_mapping = {'index': 'sessionID',
                                   'success': 'sessionSuccess',
                                   'creationTimeStamp': 'creationTimeStamp',
                                   'elements': 'elements',
                                   'tagDescriptions': 'skills',
                                   'scoring': 'scoring'}

    exercise = library_dict[exerciseID]
    sessions = exercise.keys()

    # Create basis of the question dataframe
    df_session = pd.DataFrame.from_dict(exercise, orient='index').reset_index().rename(columns=question_columns_mapping)

    # Flatten all scoring metrics
    all_scores = []

    for i in range(len(df_session)):
        scoring = df_session.iloc[i]['scoring']
        df_scoring = pd.json_normalize(scoring)
        df_scoring['sessionID'] = len(df_scoring) * [df_session.iloc[i]['sessionID']]

        all_scores.append(df_scoring)
    
    df_all_scores = pd.concat(all_scores)
    
    df_session = pd.merge(df_session, df_all_scores, on='sessionID', how='outer')

    df_skills = handle_skills(df_session)
    df_session = pd.merge(df_session, df_skills, on='sessionID', how='outer')

    df_session = df_session.drop(['elements', 'scoring', 'skills'], axis=1)

    
    df_session['exerciseID'] = len(df_session) * [exerciseID]
    df_session = df_session.iloc[:, [-1] + list(range(len(df_session.columns)-1))]
    

    list_session_frames = []
    errors = []
    # # Append the interaction dataframe to the question
    for session in sessions:

        questions = exercise[session]['elements']
        sessionID = session


        one_session = question_dataframes(questions, sessionID) 
        list_session_frames.append(one_session)
  
    df_question = pd.concat(list_session_frames, axis=0, ignore_index=True)

    
    # Join the interaction dataframe with the question dataframe
    df_session = pd.merge(df_session, df_question, on="sessionID", how='outer')
        
    
    return df_session

In [154]:
def exercise_dataframes(library_dict):
    """
    A dataframe with all sessions of one exercise and all questions of that session and all interactions of that question and all events of that interaction included.
    """

    exercises = library_dict.keys()

    all_exercise_frames = []
    for exercise in exercises:
        one_exercise = session_dataframes(library_dict, exercise)
        all_exercise_frames.append(one_exercise)

    df_exercises = pd.concat(all_exercise_frames)

    return df_exercises

In [ ]:
#df_library = exercise_dataframes(library_dict)

/var/folders/0_/xw_ks0g978df8kf5jp1rvf2h0000gn/T/ipykernel_18444/3511764737.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_scores = pd.concat(all_scores)
/var/folders/0_/xw_ks0g978df8kf5jp1rvf2h0000gn/T/ipykernel_18444/3511764737.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_scores = pd.concat(all_scores)
/var/folders/0_/xw_ks0g978df8kf5jp1rvf2h0000gn/T/ipykernel_18444/3511764737.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries

In [ ]:
#df_library.to_csv("algebrakit_firstbatch.csv", index=False)

In [ ]:
df_library = 